# 02 — Data Cleaning Pipeline
## Overview
This notebook implements the cleaning and standardization pipeline
for the higher education outcomes dataset.

Main objectives:

- standardize schemas
- normalize categorical values
- resolve structural inconsistencies
- validate metric integrity
- generate analysis-ready tables

The notebook assumes that the raw data audit performed in
`01_data_audit.ipynb` has already been completed.

In [1]:
from notebook_utils import ensure_repo_root

# Establish the repository root as the working directory for this notebook
ensure_repo_root()

WindowsPath('C:/Github/higher-education-outcomes-analysis')

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_utils import load_data

enrollment = load_data("enrollment")
programs = load_data("programs")
offering = load_data("offering")


In [3]:
from src.config.mappings import (
    CANONICAL_MAPPINGS,
    TYPO_MAPPINGS,
)

from src.cleaning import (
    drop_columns,
    normalize_text_columns,
    correct_typo_variants,
    apply_canonical_taxonomy,
    reconcile_metric_totals,
)

from src.utils.text import normalize_text

drop columns

In [4]:
enrollment_redundant_metadata_cols = [
    "shift",
    "weekday",
    "schedule_time",
    "delivery_mode",
    "campus",
]

In [5]:
enrollment_cleaned = drop_columns(
    df=enrollment,
    cols_to_drop=enrollment_redundant_metadata_cols,
    verbose=True,
)

Columns succesfully dropped: ['shift', 'weekday', 'schedule_time', 'delivery_mode', 'campus']


normalize text columns

In [6]:
text_cols = [
    "shift",
    "weekday",
    "schedule_time",
    "delivery_mode",
]

In [7]:
offering_normalized = normalize_text_columns(
    df=offering,
    text_cols=text_cols,
    normalize_text=normalize_text,
    verbose=True,
)

Text columns normalized: ['shift', 'weekday', 'schedule_time', 'delivery_mode']


before/after

In [8]:
display(offering[text_cols].nunique())
display(offering_normalized[text_cols].nunique())

shift             6
weekday          15
schedule_time    38
delivery_mode    26
dtype: int64

shift             4
weekday           8
schedule_time    34
delivery_mode    17
dtype: int64

correct typographical problems

In [9]:
offering_cleaned = correct_typo_variants(
    df=offering_normalized,
    mappings_dict=TYPO_MAPPINGS,
    unmapped="nan",
    verbose=True,
    normalize_func=None,
)

Column: 'shift'
Number of unique categories before typo correction: 4
Number of unique categories after typo correction: 3
['noche' 'manana' 'tarde']

Column: 'delivery_mode'
Number of unique categories before typo correction: 17
Number of unique categories after typo correction: 3
['presencial' nan 'virtual']

Column: 'weekday'
Number of unique categories before typo correction: 8
Number of unique categories after typo correction: 6
['martes' 'sabado' 'jueves' 'lunes' 'viernes' 'miercoles']



In [10]:
offering_normalized[offering_cleaned["delivery_mode"].isna()]["delivery_mode"].value_counts()

delivery_mode
4 hs presencial y 2 virtual              26
virtual (con encuentros presenciales)     5
4 presencial y 2 virtual                  5
presencial y 2 hs virtual                 4
4 hs presenciales y 2 virtuales           4
4 hs presencial 2 virtual                 3
4 presencial, 2 virtual                   3
2 presencial y 4 virtual                  2
4 presenciales y 2 virtuales              1
3 hs presencial 3 virtual                 1
2 hs presencial y 2 virtual               1
2 hs practicas, 4 presenciales            1
3 presencial y 3 virtual                  1
Name: count, dtype: int64

because every value is suitable to be assigned to a hybrid delivery mode (they all include on-site and online):

In [11]:
offering_cleaned["delivery_mode"] = offering_cleaned["delivery_mode"].fillna("hibrida")
offering_cleaned["delivery_mode"].unique()

array(['presencial', 'hibrida', 'virtual'], dtype=object)

canonical taxonomy

In [12]:
offering_canonical = apply_canonical_taxonomy(
    df=offering_cleaned,
    mappings_dict=CANONICAL_MAPPINGS,
    verbose=True,
)

Column: 'shift'
Unique categories after applying canonical taxonomy: ['night' 'morning' 'afternoon']

Column: 'delivery_mode'
Unique categories after applying canonical taxonomy: ['on-site' 'hybrid' 'online']

Column: 'weekday'
Unique categories after applying canonical taxonomy: ['tuesday' 'saturday' 'thursday' 'monday' 'friday' 'wednesday']



workload to int

In [ ]:
offering_canonical["workload"] = offering_canonical["workload"].astype("Int64")

metric reconciliation

In [14]:
metric_cols = [
    "dropout_count",
    "insufficient_count",
    "free_status_count",
    "promoted_completion_count",
    "regular_completion_count",
]

In [15]:
enrollment_cleaned = reconcile_metric_totals(
    df=enrollment_cleaned,
    component_columns=metric_cols,
    reported_column="total_enrollment",
    tolerance=2,
    overwrite=True,
)

keeps the reported metric, while replaces it in total_enrollment with the computed one, always that the difference<=2

In [16]:
enrollment_cleaned.loc[
    enrollment_cleaned["total_enrollment_difference"] != 0,
    [
        "reported_total_enrollment",
        "computed_total_enrollment",
        "reconciled_total_enrollment",
        "total_enrollment_difference",
    ],
]

,reported_total_enrollment,computed_total_enrollment,reconciled_total_enrollment,total_enrollment_difference
35,19,18,18,1
101,51,49,49,2
111,58,57,57,1
145,57,56,56,1


join tables

In [25]:
data = enrollment_cleaned.merge(offering_canonical, on=["course_code", "section"], how="left")
print(data.isna().any().sum())

6


A small subset of unmatched course-section observations was excluded during the cleaning stage due to unresolved operational metadata gaps after integration with the canonical Offering source.

In [22]:
data = data.dropna()
data.shape

(297, 20)

In [ ]:
data = data.merge(programs, on="program_code", how="left")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   course_name                  297 non-null    object
 1   section                      297 non-null    int64 
 2   total_enrollment             297 non-null    int64 
 3   dropout_count                297 non-null    int64 
 4   insufficient_count           297 non-null    int64 
 5   free_status_count            297 non-null    int64 
 6   promoted_completion_count    297 non-null    int64 
 7   regular_completion_count     297 non-null    int64 
 8   course_code                  297 non-null    object
 9   program_code                 297 non-null    object
 10  reported_total_enrollment    297 non-null    int64 
 11  computed_total_enrollment    297 non-null    int64 
 12  total_enrollment_difference  297 non-null    int64 
 13  reconciled_total_enrollment  297 no